# ir_calendar_consume_batches — 批次取檔 → Volume / bronze / silver / 系統 API

- 用途：把爬蟲 VM 發布在內網目錄的批次（`<root_url>/batches/<批次>/`）搬進 Databricks：實體檔案落 Volume、每個檔案原樣進 bronze、從 bronze 算出 silver、`api/*.json` 轉送系統 API。取代舊的 `01_DELL_get_finance_report_file`。
- 輸入：內網 HTTPS 目錄（只用 GET）：`health.json`、`batches/` 清單、各批的 `manifest.json` 與檔案。**不解析檔名、不掃來源目錄找檔、不猜公司**，全部靠 manifest。
- 輸出：
  - Volume：`<volume_root>/<分類目錄>/<公司代稱>/<交付檔名>`（實體檔案）；`<volume_root>/ir_calendar/batches/<批次>/`（JSON 歸檔）；`<volume_root>/ir_calendar/_cursor.json`（游標）
  - bronze：`b_{domain}_record`（append-only，一檔一列）、`b_{domain}_batch_log`
  - silver：`s_{domain}_conference`、`s_{domain}_summary`、`s_{domain}_document_file`、`s_{domain}_company`（MERGE）
  - 系統 API：`<api_base_url>/companies/sync`、`<api_base_url>/ir-conferences/sync`
- 表由 `ir_calendar_init_tables` 先建好；設計說明 `docs/20260921_ir_calendar_lakehouse_design.md`。
- 參數（widgets）：見 [c01] 與下表。
- 排程：跟著 VM 班次（掃描 08:10 / 17:10、抓檔 09:40 / 18:40 之後各一次），或每小時一次。job cluster。
- 負責人 / 更新日期：（填）/ 2026-09-21
- 規劃文件：`ir_calendar/doc/20260918_pass_data_to_databricks.md`（第 7 節 manifest、第 8 節流程、第 16–17 節檔名與公司代稱）；系統 API：`doc/20260915_API_Sepc_系統後端開發者定版.md`。

## 一次執行做什麼

1. `GET <root>/health.json`：超過 `health_max_age_hours` 沒更新或 `status = ALERT` → 記告警（批次照處理，最後讓 job 失敗）
2. `GET <root>/batches/`：序號 > 游標且有 `_SUCCESS` 的目錄才算新批次（沒 `_SUCCESS` = 還在搬，下次再看）
3. 依序號由小到大，每一批：
   1. 讀 `manifest.json`；`prev_seq` 必須等於游標，否則丟 `MissingBatch` 停住（缺批要人補搬，不能跳過）
   2. 逐檔下載、核 sha256。`ir_document_file` 落 Volume；其餘 JSON 歸檔到 `ir_calendar/batches/<批次>/`
   3. **bronze**：每個檔案一列（`payload` = JSON 全文；實體檔 = manifest 條目），連 manifest 本身也一列。先 `DELETE WHERE batch_id` 再 append，重跑冪等
   4. **silver**：只拿這一批的 bronze 列，跑 [c06] 的 transform，依 `TABLE_KEYS` MERGE 進 4 張 silver
   5. `api_payload` 檔 POST 到系統 API（`Idempotency-Key = <批次>/<endpoint>`；主檔批次 `companies/sync` 先送）。送出前依 `api_fiscal_period` 換算 `fiscalPeriod`，log 會印出換算對照
   6. 全部成功 → `batch_log` 寫 SUCCESS（含 API 回應）、游標推進到這一批；任一步失敗 → `batch_log` 寫 FAILED、游標不動、job 失敗
4. 有告警就 `raise`，讓 Databricks job 顯示失敗

**重跑安全**：游標沒推進的批次整批重來。Volume 同名覆蓋、bronze 刪後重寫、silver MERGE、API 有冪等鍵，重來不會多寫。

**重算 silver**：`rebuild_silver = true` 時不消費新批次，改成拿整張 bronze 跑 transform 再 MERGE（同鍵取最新 `seq`）。silver 改欄位、改 transform、或懷疑資料壞掉時用。

## 參數

| widget | 預設 | 說明 |
|---|---|---|
| `catalog` / `schema` | 空（必填） | Delta table 所在；對照 `config/project.yml` |
| `domain` | `ir_calendar` | 表名中段；層級前綴 `b_` / `s_` 固定 |
| `root_url` | `https://172.17.251.25/data/ir_calendar/` | 內網取檔根目錄（規劃書待決 #1，對應規則仍待內網端確認） |
| `volume_root` | `/Volumes/micenter/mi3_datahub_prod/micenterfile_ext/unstructured_data_file` | 實體檔案與歸檔根目錄 |
| `api_base_url` | 空 | 系統 API，例 `https://<host>/api/v1`。空字串 = 不轉送 |
| `api_key_secret` | 空 | `scope/key`，用 `dbutils.secrets` 取 `X-Api-Key`。空字串且 `api_base_url` 有值時不帶 key |
| `api_fiscal_period` | `quarter` | POST 前怎麼處理 body 的 `fiscalPeriod`（dropdown）：`quarter` = 只留季別（`2026Q3` → `Q3`）／`null` = 一律清成 null／`as_is` = 照爬蟲原值。只影響送出去的 body，bronze / silver 永遠存原值 |
| `verify_ssl` | `false` | 內網自簽憑證 |
| `dry_run` | `false` | `true`：只印會做什麼，不寫 Volume、不寫表、不打 API、不推游標 |
| `write_tables` | `true` | `false`：跳過 bronze / silver / batch_log 寫入（表還沒建好時的過渡用） |
| `rebuild_silver` | `false` | `true`：不消費新批次，從整張 bronze 重算 silver |
| `max_batches` | `50` | 單次最多處理幾批；首次追趕可調大 |
| `health_max_age_hours` | `24` | `health.json` 逾時門檻 |
| `job_run_id` | 空 | job parameters 填 `{{job.run_id}}`，寫進 `batch_log` 方便對 log |

## 維護者須知

| 要改什麼 | 改哪裡 |
|---|---|
| Volume 分類目錄名 | [c02] `CATEGORY_DIR`，只改這一處 |
| manifest 沒帶 endpoint 時的備援對應 | [c02] `ENDPOINT_BY_PATH` |
| 系統端要哪種 `fiscalPeriod` | 先用 widget `api_fiscal_period` 切（`quarter` / `null` / `as_is`）；要加第四種才改 [c01] `FISCAL_PERIOD_MODES` + [c03] `fiscal_period_for_api` |
| 爬蟲多了欄位 | bronze 不用改。silver 要用它：建表 notebook `ALTER TABLE ADD COLUMNS` → [c04] 對應的 `*_PAYLOAD` schema 加欄位 → [c06] transform 的 `select` 加欄位 |
| 爬蟲多了 record_type | bronze 自動收。要建 silver：建表 notebook 加 DDL → [c04] `TABLE_KEYS` + payload schema → [c06] 加 `transform_<短名>` 並登記 `SILVER_TRANSFORMS` |
| 時間處理 | [c04] `ts_col`：帶時區照用、不帶時區補 `+08:00`（台北），存成 UTC 瞬間。`date_col` 只取前 10 碼解析，**不做時區處理**：`conference_date` 是台北曆日，爬蟲端已換算 |
| 游標 | 存在 Volume 的 `_cursor.json`，不在 Delta：不依賴表已建好，也方便人工改（補批次後把 `last_seq` 改回去即可重處理） |
| 只想本機測邏輯 | [c02]～[c06] 沒有 I/O、沒有 `spark` 全域變數 / `dbutils`；`tests/test_ir_calendar_consume.py` 用 `nbload.load_cells` 載入，用本機 pyspark 跑 transform |

## 首次上線順序

1. 內網端確認 `root_url` 真的對到 VM 的 `out/`（`GET <root>/health.json` 有東西）
2. 跑 `ir_calendar_init_tables` 建表
3. `dry_run = true` 跑一次，看列出的批次、落地路徑、bronze 列數對不對
4. `dry_run = false`、`api_base_url` 留空跑一次：落 Volume + bronze + silver，確認目錄與表內容
5. 填 `api_base_url` 與 `api_key_secret` 再跑；系統 API 應回 `created`，再跑一次應回 `unchanged`
6. 設排程與 job parameters（`catalog`、`schema`、`job_run_id = {{job.run_id}}`）


In [ ]:
# [c01] params
# 預設值取自 config/project.yml（catalog / schema）；job 執行時由 job parameters 覆蓋。
# 注意：widget 一旦在 notebook 建立過，改 code 的預設值不會更新既有 widget；要重設請先 dbutils.widgets.removeAll() 再跑本 cell。布林 / 數字都用字串填，這裡統一轉型。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")     # 表名中段；層級前綴 b_ / s_ 固定，見 docs/conventions.md 2.1
dbutils.widgets.text("root_url", "https://172.17.251.25/data/ir_calendar/")
dbutils.widgets.text("volume_root", "/Volumes/micenter/mi3_datahub_prod/micenterfile_ext/unstructured_data_file")
dbutils.widgets.text("api_base_url", "")          # 空 = 不轉送系統 API
dbutils.widgets.text("api_key_secret", "")        # "scope/key"；憑證只走 dbutils.secrets
dbutils.widgets.text("verify_ssl", "false")       # 內網自簽憑證
dbutils.widgets.text("dry_run", "false")
dbutils.widgets.text("write_tables", "true")
dbutils.widgets.text("rebuild_silver", "false")   # true：不消費新批次，從整張 bronze 重算 silver
dbutils.widgets.text("max_batches", "50")
dbutils.widgets.text("health_max_age_hours", "24")
# 送給系統 API 的 fiscalPeriod 要長怎樣（值還在試，用 dropdown 直接切，不用改 code）：
#   quarter = 只留季別（2026Q3 → Q3）／null = 一律清成 null／as_is = 照爬蟲原值送
# 只影響 POST 出去的 body；bronze 與 silver 存的永遠是爬蟲原值。
# 模式的實作在 [c03] fiscal_period_for_api，要加第四種兩邊都要改
FISCAL_PERIOD_MODES = ("quarter", "null", "as_is")
dbutils.widgets.dropdown("api_fiscal_period", "quarter", list(FISCAL_PERIOD_MODES))
dbutils.widgets.text("job_run_id", "")            # job parameters 填 {{job.run_id}}


def _flag(name: str) -> bool:
    return dbutils.widgets.get(name).strip().lower() == "true"


settings = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "domain": dbutils.widgets.get("domain").strip(),
    "root_url": dbutils.widgets.get("root_url").strip().rstrip("/") + "/",
    "volume_root": dbutils.widgets.get("volume_root").strip().rstrip("/"),
    "api_base_url": dbutils.widgets.get("api_base_url").strip().rstrip("/"),
    "api_key_secret": dbutils.widgets.get("api_key_secret").strip(),
    "api_fiscal_period": dbutils.widgets.get("api_fiscal_period").strip(),
    "verify_ssl": _flag("verify_ssl"),
    "dry_run": _flag("dry_run"),
    "write_tables": _flag("write_tables"),
    "rebuild_silver": _flag("rebuild_silver"),
    "max_batches": int(dbutils.widgets.get("max_batches")),
    "health_max_age_hours": float(dbutils.widgets.get("health_max_age_hours")),
    "job_run_id": dbutils.widgets.get("job_run_id").strip() or None,
}
assert settings["catalog"] and settings["schema"] and settings["domain"], "catalog / schema / domain 不可為空"
assert settings["volume_root"].startswith("/Volumes/"), "volume_root 必須是 UC Volume 路徑"
assert settings["api_fiscal_period"] in FISCAL_PERIOD_MODES, (
    f"api_fiscal_period 只能是 {FISCAL_PERIOD_MODES}")
print({k: v for k, v in settings.items() if k != "api_key_secret"})


In [ ]:
# [c02] imports
# 共用 import 與常數。本 cell 與 [c03]～[c06] 不碰 spark 全域變數 / dbutils / 網路，可被 tests/ 載入。
import hashlib
import json
import os
import re
from datetime import datetime, timedelta, timezone
from html import unescape
from urllib.parse import unquote

from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F

# 公司分類 → Volume 子目錄。分類值來自爬蟲主檔（category / category_name），manifest 每個實體檔都帶。
CATEGORY_DIR = {
    "PANEL_PEER": "panel_peer",       # 面板同業
    "CUSTOMER": "brand_customer",     # 品牌客戶
    "SUPPLIER": "supplier",           # 供應商
}
UNKNOWN_CATEGORY_DIR = "uncategorized"    # 分類未知（不該發生，只是不讓檔案掉在根目錄）
UNKNOWN_SLUG_DIR = "_unknown_company"     # 缺代稱（爬蟲端缺代稱時整批不發，這裡只是保底）
CONTROL_DIR = "ir_calendar"               # <volume_root>/ir_calendar/：游標、批次歸檔（JSON）

# manifest 沒帶 endpoint 時的備援對應（batch.py 都有帶，這裡只防舊批次）
ENDPOINT_BY_PATH = {
    "api/ir_conferences_sync.json": "ir-conferences/sync",
    "api/companies_sync.json": "companies/sync",
}
TAIPEI = timezone(timedelta(hours=8))     # 爬蟲不帶時區的時間字串都是台北時間
UTC = timezone.utc  # noqa: UP017 - 本機測試仍是 Python 3.10，不用 datetime.UTC

# 表命名：<層級前綴><domain>_<短名>，層級前綴固定（docs/conventions.md 2.1）
LAYER_PREFIX = {"bronze": "b_", "silver": "s_"}

BATCH_RE = re.compile(r"^(\d{6})_\d{8}T\d{6}$")
FISCAL_PERIOD_FIELD = "fiscalPeriod"            # API body 裡的期別欄位；送出前依 [c01] 的模式換算
QUARTER_RE = re.compile(r"Q([1-4])", re.IGNORECASE)   # 2026Q3 / FY2026Q3 / Q3 都取得到季別
_HREF_RE = re.compile(r"""href\s*=\s*["']([^"']+)["']""", re.IGNORECASE)


In [ ]:
# [c03] pure_helpers
# 純函式：目錄索引解析、批次篩選、路徑組合、時間判斷。無 I/O。


class MissingBatch(Exception):
    """搬檔漏了一批：manifest 的 prev_seq 對不上游標。停住等人補，不跳過（規劃書 G6）。"""


class IntegrityError(Exception):
    """下載內容的 sha256 與 manifest 不符。"""


def parse_listing(html: str) -> list[str]:
    """Apache / Nginx 式目錄索引 → 項目名稱（目錄不帶尾斜線）。去掉上層連結、排序連結與 query。"""
    names: list[str] = []
    for href in _HREF_RE.findall(html or ""):
        href = unescape(href).strip()
        if not href or href.startswith(("?", "#", "/", "..")) or "://" in href:
            continue
        href = href.split("?", 1)[0].split("#", 1)[0]
        name = unquote(href.rstrip("/"))
        if name and "/" not in name and name not in names:
            names.append(name)
    return names


def pick_new_batches(names, last_seq: int) -> list[tuple[int, str]]:
    """目錄名符合批次格式且序號 > 游標的，依序號由小到大。"""
    out = []
    for n in names:
        m = BATCH_RE.match(n)
        if m and int(m.group(1)) > last_seq:
            out.append((int(m.group(1)), n))
    return sorted(out)


def check_prev_seq(manifest: dict, last_seq: int) -> None:
    prev = manifest.get("prev_seq")
    if (prev or 0) != last_seq:
        raise MissingBatch(f"批次 {manifest.get('batch_id')} 的 prev_seq={prev}，"
                           f"但游標在 {last_seq}：中間缺了批次，請先補搬")


def volume_target(volume_root: str, f: dict) -> str:
    """實體檔案的落地路徑：<root>/<分類目錄>/<公司代稱>/<交付檔名>。分類與代稱都取 manifest 欄位，不拆檔名。"""
    cat_dir = CATEGORY_DIR.get(f.get("category") or "", UNKNOWN_CATEGORY_DIR)
    slug = f.get("company_slug") or UNKNOWN_SLUG_DIR
    return f"{volume_root}/{cat_dir}/{slug}/{os.path.basename(f['path'])}"


def archive_target(volume_root: str, batch_id: str, rel_path: str) -> str:
    return f"{volume_root}/{CONTROL_DIR}/batches/{batch_id}/{rel_path}"


def cursor_path(volume_root: str) -> str:
    return f"{volume_root}/{CONTROL_DIR}/_cursor.json"


def is_stale(generated_at: str | None, now: datetime, max_age_hours: float) -> bool:
    if not generated_at:
        return True
    ts = datetime.fromisoformat(generated_at)
    if ts.tzinfo is None:
        ts = ts.replace(tzinfo=TAIPEI)
    return now - ts > timedelta(hours=max_age_hours)


def endpoint_for(f: dict) -> str | None:
    return f.get("endpoint") or ENDPOINT_BY_PATH.get(f["path"])


def fiscal_period_for_api(value: str | None, mode: str) -> str | None:
    """POST 出去的 fiscalPeriod 值。系統端要哪一種還在試，所以用 [c01] 的 api_fiscal_period 切換。

    quarter：只留季別，2026Q3 / FY2026Q3 / Q3 → Q3；取不到季別就送 None（不亂猜）
    null   ：一律 None（這個欄位暫不傳給系統）
    as_is  ：照爬蟲原值送
    """
    if mode == "as_is":
        return value
    if mode == "null":
        return None
    if mode != "quarter":
        raise ValueError(f"不認得的 api_fiscal_period：{mode}")
    m = QUARTER_RE.search(str(value)) if value is not None else None
    return f"Q{m.group(1)}" if m else None


def apply_fiscal_period(payload: dict, mode: str) -> tuple[dict, dict]:
    """換算 body 裡每一列的 fiscalPeriod，回 (要送出的 body, {原值: 送出值} 對照，沒變就空的)。

    不改傳進來的 payload：bronze 收的是 api/*.json 原文，只有 POST 用的這份被換過。
    沒有這個欄位的 body（companies/sync）原樣回傳。
    """
    rows = payload.get("rows")
    if mode == "as_is" or not isinstance(rows, list):
        return payload, {}
    out, changed = [], {}
    for row in rows:
        if isinstance(row, dict) and FISCAL_PERIOD_FIELD in row:
            old = row[FISCAL_PERIOD_FIELD]
            new = fiscal_period_for_api(old, mode)
            if new != old:
                changed[old] = new
                row = {**row, FISCAL_PERIOD_FIELD: new}
        out.append(row)
    return ({**payload, "rows": out}, changed) if changed else (payload, {})


In [ ]:
# [c04] table_schemas
# 三種 schema：
#   BRONZE_SCHEMA / BATCH_LOG_SCHEMA：createDataFrame 用，欄位名必須與建表 notebook 的 DDL 一致
#   *_PAYLOAD：from_json 解析 bronze.payload 用，只列 silver 會用到的欄位（爬蟲多的欄位自動忽略、少的補 NULL）
#   TABLE_KEYS：silver 的 MERGE 鍵；batch_log 也用 MERGE（job 遙測允許覆蓋）
# 另有兩個欄位轉換 helper：ts_col（字串 → UTC TIMESTAMP）、date_col（字串 → DATE，不做時區處理）。
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    IntegerType,
    LongType,
    MapType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

TABLE_KEYS: dict[str, list[str]] = {
    "batch_log": ["batch_id"],
    "conference": ["company_key", "period", "revision"],
    "summary": ["company_key", "period"],
    "document_file": ["volume_path"],
    "company": ["company_key"],
}


def _s(name: str, dtype=None, nullable: bool = True) -> StructField:
    return StructField(name, dtype or StringType(), nullable)


_ARR = ArrayType(StringType())

BRONZE_SCHEMA = StructType([
    _s("batch_id", nullable=False), _s("seq", IntegerType(), False), _s("record_type", nullable=False),
    _s("batch_path", nullable=False), _s("company_key"), _s("period"), _s("sha256"), _s("bytes", LongType()),
    _s("volume_path"), _s("payload", nullable=False), _s("ingested_at", TimestampType(), False),
])

API_RESULT_SCHEMA = StructType([
    _s("endpoint"), _s("idempotency_key"), _s("rows", IntegerType()), _s("http_status", IntegerType()),
    _s("success", BooleanType()), _s("created", IntegerType()), _s("updated", IntegerType()),
    _s("unchanged", IntegerType()), _s("failed", IntegerType()), _s("failures_json"), _s("response_json"),
    _s("posted_at", TimestampType()),
])

BATCH_LOG_SCHEMA = StructType([
    _s("batch_id", nullable=False), _s("seq", IntegerType(), False), _s("prev_seq", IntegerType()),
    _s("kind"), _s("producer_mode"), _s("producer_host"), _s("producer_git_sha"),
    _s("generated_at", TimestampType()), _s("counts", MapType(StringType(), IntegerType())),
    _s("files_total", IntegerType()), _s("landed_files", IntegerType()), _s("archived_files", IntegerType()),
    _s("bronze_rows", IntegerType()), _s("api_results", ArrayType(API_RESULT_SCHEMA)),
    _s("status", nullable=False), _s("error"), _s("job_run_id"), _s("processed_at", TimestampType(), False),
])

# ---- payload schema（爬蟲 JSON 的形狀；時間 / 日期先當字串，交給 ts_col / date_col）----
CONFERENCE_PAYLOAD = StructType([
    _s("company_key"), _s("company_name"), _s("english_name"), _s("stock_code"), _s("market"), _s("category"),
    _s("period"), _s("fiscal_period"), _s("source_fiscal_period"), _s("conference_date"), _s("start_time"),
    _s("end_time"), _s("conference_type"), _s("location"), _s("meeting_link"), _s("document_url"), _s("status"),
    _s("importance", IntegerType()), _s("source"), _s("source_url"), _s("confidence"), _s("remark"),
    _s("recipients", _ARR), _s("revision", IntegerType()), _s("crawled_at"),
])

SUMMARY_PAYLOAD = StructType([
    _s("company_key"), _s("company_name"), _s("stock_code"), _s("market"), _s("category"), _s("period"),
    _s("fiscal_period"), _s("conference_date"), _s("fallback", BooleanType()), _s("recipients", _ARR),
    _s("crawled_at"),
    _s("summary", StructType([
        _s("found", BooleanType()), _s("core_points", _ARR), _s("guidance"), _s("key_numbers", _ARR),
        _s("risks", _ARR), _s("notes"),
        _s("sources", ArrayType(StructType([_s("title"), _s("url"), _s("media")]))),
    ])),
])

# manifest.files[] 中 ir_document_file 那一條（bronze 的 payload 就是這條目）
DOCUMENT_FILE_PAYLOAD = StructType([
    _s("path"), _s("sha256"), _s("bytes", LongType()), _s("company_key"), _s("company_slug"), _s("category"),
    _s("period"), _s("doc_kind"), _s("fiscal_label"), _s("source_file"), _s("source_url"), _s("doc_date"),
])

# master/company.json 整份（rows[] 展開）
COMPANY_PAYLOAD = StructType([
    _s("generated_at"),
    _s("rows", ArrayType(StructType([
        _s("company_key"), _s("company_name"), _s("english_name"), _s("stock_code"), _s("market"),
        _s("market_type"), _s("category_name"), _s("industry"), _s("website_url"), _s("ir_url"),
        _s("aliases", _ARR), _s("recipients", _ARR), _s("remark"), _s("is_active", BooleanType()),
        _s("profile_source"), _s("file_slug"),
    ]))),
])


def ts_col(c: Column) -> Column:
    """ISO 字串 → TIMESTAMP（UTC 瞬間）。帶時區（Z / +08:00）照用；不帶時區視為台北時間，補 +08:00 再 cast。
    補偏移量的作法不受 spark.sql.session.timeZone 影響。空字串視為 NULL；格式錯誤在 ANSI 模式會拋錯（來源都是 ISO，故意不放寬）。"""
    s = F.nullif(F.trim(c), F.lit(""))
    fixed = F.when(s.rlike(r"(Z|[+-]\d{2}:?\d{2})$"), s).otherwise(F.concat(s, F.lit("+08:00")))
    return fixed.cast("timestamp")


def date_col(c: Column) -> Column:
    """YYYY-MM-DD（或更長的 ISO 字串取前 10 碼）→ DATE。不做時區處理：法說日期由爬蟲端統一換算成台北曆日。"""
    return F.to_date(F.nullif(F.substring(F.trim(c), 1, 10), F.lit("")), "yyyy-MM-dd")


In [ ]:
# [c05] row_builders
# bronze 與 batch_log 的列（dict，鍵 = 欄位名）。純函式，無 I/O；createDataFrame 在 [c09]。
# 只有這兩種列由 Python 組：bronze 是原樣落地，batch_log 是 job 自己的紀錄。silver 全部由 [c06] 的 DataFrame 轉換產生。


def to_ts(s: str | None) -> datetime | None:
    """ISO 字串 → 帶時區 datetime（UTC）。不帶時區視為台北時間；空值回 None。只用在 batch_log 的 manifest.generated_at。"""
    if not s:
        return None
    ts = datetime.fromisoformat(s.replace("Z", "+00:00"))
    if ts.tzinfo is None:
        ts = ts.replace(tzinfo=TAIPEI)
    return ts.astimezone(UTC)


def to_int(v) -> int | None:
    return None if v is None or v == "" else int(v)


def js(obj) -> str | None:
    """物件 → JSON 字串原文；None 保持 None。"""
    return None if obj is None else json.dumps(obj, ensure_ascii=False)


def bronze_row(manifest: dict, f: dict, payload: str, *, volume_path: str | None, now: datetime) -> dict:
    """manifest.files[] 的一個條目 → b_*_record 一列。payload 是 JSON 全文（實體檔則傳 manifest 條目的 JSON）。"""
    return {
        "batch_id": manifest["batch_id"], "seq": int(manifest["seq"]), "record_type": f.get("record_type") or "unknown",
        "batch_path": f["path"], "company_key": f.get("company_key"), "period": f.get("period"),
        "sha256": f.get("sha256"), "bytes": to_int(f.get("bytes")), "volume_path": volume_path,
        "payload": payload, "ingested_at": now,
    }


def manifest_bronze_row(manifest: dict, *, now: datetime) -> dict:
    """manifest 本身也進 bronze 一列（record_type = batch_manifest），整批的血緣不必回 Volume 找。"""
    f = {"path": "manifest.json", "record_type": "batch_manifest"}
    return bronze_row(manifest, f, js(manifest), volume_path=None, now=now)


def api_result(*, endpoint: str, idem_key: str, rows: int, http_status: int | None, body: dict | None,
               now: datetime) -> dict:
    """一次 POST 的結果，對應 batch_log.api_results 的 struct。body 為系統 API 的 envelope。"""
    data = (body or {}).get("data") or {}
    return {
        "endpoint": endpoint, "idempotency_key": idem_key, "rows": rows, "http_status": http_status,
        "success": data.get("success"), "created": to_int(data.get("created")), "updated": to_int(data.get("updated")),
        "unchanged": to_int(data.get("unchanged")), "failed": to_int(data.get("failed")),
        "failures_json": js(data.get("failures")), "response_json": (js(body) or "")[:4000] or None, "posted_at": now,
    }


def batch_log_row(manifest: dict, *, status: str, now: datetime, landed: int = 0, archived: int = 0,
                  bronze_rows: int = 0, api_results: list[dict] | None = None, error: str | None = None,
                  job_run_id: str | None = None) -> dict:
    p = manifest.get("producer") or {}
    return {
        "batch_id": manifest["batch_id"], "seq": int(manifest["seq"]), "prev_seq": to_int(manifest.get("prev_seq")),
        "kind": manifest.get("kind"), "producer_mode": p.get("mode"), "producer_host": p.get("host"),
        "producer_git_sha": p.get("git_sha"), "generated_at": to_ts(manifest.get("generated_at")),
        "counts": {k: int(v) for k, v in (manifest.get("counts") or {}).items()},
        "files_total": len(manifest.get("files") or []), "landed_files": landed, "archived_files": archived,
        "bronze_rows": bronze_rows, "api_results": list(api_results or []),
        "status": status, "error": str(error)[:2000] if error else None,
        "job_run_id": job_run_id, "processed_at": now,
    }


In [ ]:
# [c06] transform_silver
# bronze → silver 的純函式 DataFrame -> DataFrame。輸入是 b_*_record 形狀（至少 record_type / batch_id / seq / batch_path /
# volume_path / payload），輸出是 silver 表的欄位（不含 updated_at，由 [c09] 寫入時補）。全部用 F.*，本機可測。
# 加 silver 表：寫 transform_<短名>，登記到 SILVER_TRANSFORMS，鍵放 [c04] TABLE_KEYS，DDL 放建表 notebook。

def _lineage() -> list[Column]:
    """bronze 血緣欄位，每張 silver 都帶。做成函式：模組層級不建 Column，cell 載入時不需要 SparkContext。"""
    return [F.col("batch_id"), F.col("seq"), F.col("batch_path")]


def _parsed(df: DataFrame, record_type: str, schema) -> DataFrame:
    """篩出一種 record_type，payload 解析成 struct 欄 p。"""
    return df.filter(F.col("record_type") == record_type).withColumn("p", F.from_json(F.col("payload"), schema))


def latest_per_key(df: DataFrame, keys: list[str]) -> DataFrame:
    """同鍵多列時只留 seq 最大的（批次越新越大）；同批同鍵再以 batch_path 決定，保證確定性。MERGE 要求來源鍵唯一。"""
    w = Window.partitionBy(*keys).orderBy(F.col("seq").desc(), F.col("batch_path").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")


def transform_conference(df: DataFrame) -> DataFrame:
    """record_type = ir_conference（calendar/*.json，一檔 = 一場次的一個 revision）。"""
    d = _parsed(df, "ir_conference", CONFERENCE_PAYLOAD)
    return d.select(
        F.col("p.company_key").alias("company_key"), F.col("p.company_name").alias("company_name"),
        F.col("p.english_name").alias("english_name"), F.col("p.stock_code").alias("stock_code"),
        F.col("p.market").alias("market"), F.col("p.category").alias("category"),
        F.col("p.period").alias("period"), F.col("p.fiscal_period").alias("fiscal_period"),
        F.col("p.source_fiscal_period").alias("source_fiscal_period"),
        date_col(F.col("p.conference_date")).alias("conference_date"),
        F.col("p.start_time").alias("start_time"), F.col("p.end_time").alias("end_time"),
        F.col("p.conference_type").alias("conference_type"), F.col("p.location").alias("location"),
        F.col("p.meeting_link").alias("meeting_link"), F.col("p.document_url").alias("document_url"),
        F.col("p.status").alias("status"), F.col("p.importance").alias("importance"),
        F.col("p.source").alias("source"), F.col("p.source_url").alias("source_url"),
        F.col("p.confidence").alias("confidence"), F.col("p.remark").alias("remark"),
        F.col("p.recipients").alias("recipients"),
        F.coalesce(F.col("p.revision"), F.lit(0)).alias("revision"),
        ts_col(F.col("p.crawled_at")).alias("crawled_at"),
        *_lineage(),
    )


def transform_summary(df: DataFrame) -> DataFrame:
    """record_type = ir_summary（summaries/*.json）。summary 物件展開常用欄位，另以 get_json_object 保留原文。"""
    d = _parsed(df, "ir_summary", SUMMARY_PAYLOAD)
    return d.select(
        F.col("p.company_key").alias("company_key"), F.col("p.company_name").alias("company_name"),
        F.col("p.stock_code").alias("stock_code"), F.col("p.market").alias("market"),
        F.col("p.category").alias("category"), F.col("p.period").alias("period"),
        F.col("p.fiscal_period").alias("fiscal_period"),
        date_col(F.col("p.conference_date")).alias("conference_date"),
        F.col("p.fallback").alias("fallback"), F.col("p.summary.found").alias("found"),
        F.col("p.summary.core_points").alias("core_points"), F.col("p.summary.guidance").alias("guidance"),
        F.col("p.summary.key_numbers").alias("key_numbers"), F.col("p.summary.risks").alias("risks"),
        F.col("p.summary.notes").alias("notes"), F.col("p.summary.sources").alias("sources"),
        F.get_json_object(F.col("payload"), "$.summary").alias("summary_json"),
        F.col("p.recipients").alias("recipients"),
        ts_col(F.col("p.crawled_at")).alias("crawled_at"),
        *_lineage(),
    )


def transform_document_file(df: DataFrame) -> DataFrame:
    """record_type = ir_document_file（實體檔；payload 是 manifest 條目，volume_path 是落地路徑）。不拆檔名。"""
    d = _parsed(df, "ir_document_file", DOCUMENT_FILE_PAYLOAD)
    return d.select(
        F.col("volume_path").alias("volume_path"),
        F.element_at(F.split(F.col("batch_path"), "/"), -1).alias("file_name"),
        F.col("p.company_key").alias("company_key"), F.col("p.company_slug").alias("company_slug"),
        F.col("p.category").alias("category"), F.col("p.period").alias("period"),
        F.col("p.doc_kind").alias("doc_kind"), F.col("p.fiscal_label").alias("fiscal_label"),
        F.col("p.sha256").alias("sha256"), F.col("p.bytes").alias("bytes"),
        F.col("p.source_file").alias("source_file"), F.col("p.source_url").alias("source_url"),
        date_col(F.col("p.doc_date")).alias("doc_date"),
        *_lineage(),
    )


def transform_company(df: DataFrame) -> DataFrame:
    """record_type = app_company（主檔批次 master/company.json 整份，rows[] 展開）。"""
    d = _parsed(df, "app_company", COMPANY_PAYLOAD)
    r = F.col("r")
    return d.select(F.explode(F.col("p.rows")).alias("r"), F.col("p.generated_at").alias("_gen"), *_lineage()).select(
        r["company_key"].alias("company_key"), r["company_name"].alias("company_name"),
        r["english_name"].alias("english_name"), r["stock_code"].alias("stock_code"), r["market"].alias("market"),
        r["market_type"].alias("market_type"), r["category_name"].alias("category_name"),
        r["industry"].alias("industry"), r["website_url"].alias("website_url"), r["ir_url"].alias("ir_url"),
        r["aliases"].alias("aliases"), r["recipients"].alias("recipients"), r["remark"].alias("remark"),
        r["is_active"].alias("is_active"), r["profile_source"].alias("profile_source"),
        r["file_slug"].alias("file_slug"),
        ts_col(F.col("_gen")).alias("master_generated_at"),
        *_lineage(),
    )


# 短名 → transform。順序 = 寫入順序（主檔先於場次，與 API 順序一致）。
SILVER_TRANSFORMS = {
    "company": transform_company,
    "conference": transform_conference,
    "summary": transform_summary,
    "document_file": transform_document_file,
}


In [ ]:
# [c07] io_source_cursor
# I/O：內網目錄（只用 GET）、游標檔、Volume 寫檔。requests / urllib3 為 DBR 內建。
import requests
import urllib3


class Source:
    """內網目錄資源：只用 GET。"""

    def __init__(self, root_url: str, verify_ssl: bool):
        self.root = root_url
        self.s = requests.Session()
        self.s.verify = verify_ssl
        if not verify_ssl:
            urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    def _get(self, rel: str, **kw) -> requests.Response:
        r = self.s.get(self.root + rel, timeout=kw.pop("timeout", 120), **kw)
        r.raise_for_status()
        return r

    def listing(self, rel_dir: str) -> list[str]:
        return parse_listing(self._get(rel_dir.rstrip("/") + "/").text)

    def json(self, rel: str) -> dict:
        return self._get(rel).json()

    def bytes(self, rel: str, sha256: str | None = None) -> bytes:
        data = self._get(rel).content
        if sha256 and hashlib.sha256(data).hexdigest() != sha256:
            raise IntegrityError(f"{rel}: sha256 不符（搬檔可能壞了）")
        return data

    def exists(self, rel: str) -> bool:
        try:
            r = self.s.get(self.root + rel, timeout=30, stream=True)
            r.close()
            return r.status_code == 200
        except requests.RequestException:
            return False


class Cursor:
    """游標 = 上次成功處理的序號。存在 Volume（不是 Delta）：job 重跑或換 cluster 都拿得到，也不依賴表已建好。"""

    def __init__(self, volume_root: str):
        self.path = cursor_path(volume_root)

    def load(self) -> dict:
        try:
            with open(self.path, encoding="utf-8") as f:
                return json.load(f)
        except (OSError, json.JSONDecodeError):
            return {"last_seq": 0, "last_batch_id": None}

    def save(self, seq: int, batch_id: str) -> None:
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        tmp = self.path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump({"last_seq": seq, "last_batch_id": batch_id,
                       "updated_at": datetime.now(UTC).isoformat(timespec="seconds")}, f)
        os.replace(tmp, self.path)


def write_bytes(target: str, data: bytes, dry_run: bool) -> None:
    if dry_run:
        print(f"    [dry-run] 會寫入 {target}（{len(data) / 1024:.1f} KB）")
        return
    os.makedirs(os.path.dirname(target), exist_ok=True)
    tmp = target + ".tmp"
    with open(tmp, "wb") as f:
        f.write(data)
    os.replace(tmp, target)      # 同名即覆蓋：同一場次的新 revision 取代舊檔


In [ ]:
# [c08] io_api
# 轉送系統 API。api_base_url 為空時略過；API Key 只從 dbutils.secrets 取，不出現在 code 或 widget。
# 送出前一律過 [c03] apply_fiscal_period：依 api_fiscal_period 換算 body 的 fiscalPeriod
# （bronze 存的仍是爬蟲原文）。


def api_key(settings: dict) -> str | None:
    ref = settings.get("api_key_secret") or ""
    if "/" not in ref:
        return None
    scope, key = ref.split("/", 1)
    return dbutils.secrets.get(scope=scope, key=key)


def api_post(settings: dict, endpoint: str, payload: dict, idem_key: str) -> tuple[int | None, dict | None]:
    """回 (http_status, body)。未實際送出（未設 api_base_url / dry_run）回 (None, None)。"""
    base = settings["api_base_url"]
    mode = settings["api_fiscal_period"]
    payload, changed = apply_fiscal_period(payload, mode)
    n_rows = len(payload.get("rows", []))
    if changed:
        pairs = "、".join(f"{k}→{v}" for k, v in list(changed.items())[:5])
        more = f" 等 {len(changed)} 種" if len(changed) > 5 else ""
        print(f"    fiscalPeriod（{mode}）：{pairs}{more}")
    if not base:
        print(f"    api_base_url 未設定，略過轉送 {endpoint}（{n_rows} 列）")
        return None, None
    if settings["dry_run"]:
        print(f"    [dry-run] 會 POST {base}/{endpoint}（{n_rows} 列）")
        return None, None
    headers = {"Content-Type": "application/json", "Idempotency-Key": idem_key}
    key = api_key(settings)
    if key:
        headers["X-Api-Key"] = key
    r = requests.post(f"{base}/{endpoint}", json=payload, headers=headers,
                      timeout=300, verify=settings["verify_ssl"])
    r.raise_for_status()
    body = r.json() if r.content else {}
    print(f"    POST {endpoint}: HTTP {r.status_code} {json.dumps(body, ensure_ascii=False)[:300]}")
    return r.status_code, body


In [ ]:
# [c09] io_delta
# Delta 寫入三種：
#   write_bronze   ：b_*_record，先 DELETE WHERE batch_id 再 append（重跑冪等，不留重複列）
#   write_silver   ：對 df_bronze 跑 [c06] 每個 transform → latest_per_key → MERGE（whenMatched 全欄更新 / whenNotMatched 插入）
#   write_batch_log：b_*_batch_log，以 batch_id MERGE
# dry_run 或 write_tables = false 時只印列數。表不存在會直接拋錯：先跑 ir_calendar_init_tables。
from delta.tables import DeltaTable


def table_name(settings: dict, layer: str, short: str) -> str:
    return f"{settings['catalog']}.{settings['schema']}.{LAYER_PREFIX[layer]}{settings['domain']}_{short}"


def _skip(settings: dict, what: str, n: int) -> bool:
    if settings["dry_run"] or not settings["write_tables"]:
        print(f"    [skip] {what}: {n} 列（dry_run 或 write_tables=false）")
        return True
    return False


def _merge(table: str, df: DataFrame, keys: list[str]) -> None:
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    (DeltaTable.forName(spark, table).alias("t")
     .merge(df.alias("s"), cond)
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())


def write_bronze(settings: dict, batch_id: str, rows: list[dict]) -> int:
    table = table_name(settings, "bronze", "record")
    if not rows or _skip(settings, table, len(rows)):
        return 0
    df = spark.createDataFrame(rows, schema=BRONZE_SCHEMA)
    DeltaTable.forName(spark, table).delete(F.col("batch_id") == F.lit(batch_id))   # 重跑：先清同批舊列
    df.write.format("delta").mode("append").saveAsTable(table)
    print(f"    {table}: append {len(rows)} 列（batch {batch_id}）")
    return len(rows)


def write_silver(settings: dict, df_bronze: DataFrame) -> dict[str, int]:
    """df_bronze 可以是一批（增量）或整張表（rebuild）。回各表寫入列數。"""
    out = {}
    for short, transform in SILVER_TRANSFORMS.items():
        table = table_name(settings, "silver", short)
        keys = TABLE_KEYS[short]
        df = latest_per_key(transform(df_bronze), keys).withColumn("updated_at", F.current_timestamp())
        n = df.count()          # silver 都是小表，count 可接受；順便讓空結果不必跑 MERGE
        out[short] = n
        if n == 0 or _skip(settings, table, n):
            continue
        _merge(table, df, keys)
        print(f"    {table}: merge {n} 列")
    return out


def write_batch_log(settings: dict, row: dict) -> None:
    table = table_name(settings, "bronze", "batch_log")
    if _skip(settings, table, 1):
        return
    _merge(table, spark.createDataFrame([row], schema=BATCH_LOG_SCHEMA), TABLE_KEYS["batch_log"])


In [ ]:
# [c10] process_batch
# 處理一批：下載 → 落 Volume / 歸檔 → bronze append → silver MERGE → 轉送 API。
# 任何一步失敗就丟例外，游標不推進，下次從同一批重來。


def process_batch(src: Source, settings: dict, name: str, last_seq: int, now: datetime) -> dict:
    manifest = src.json(f"batches/{name}/manifest.json")
    check_prev_seq(manifest, last_seq)
    vol, dry = settings["volume_root"], settings["dry_run"]
    summary = {"batch_id": name, "kind": manifest.get("kind"), "counts": manifest.get("counts"),
               "landed": 0, "archived": 0, "bronze_rows": 0, "silver": {}, "api": [], "manifest": manifest}
    print(f"  {name}: kind={manifest.get('kind')} mode={(manifest.get('producer') or {}).get('mode')} "
          f"counts={manifest.get('counts')}")

    bronze: list[dict] = [manifest_bronze_row(manifest, now=now)]
    payloads: list[tuple[dict, dict]] = []
    # 主檔批次先送公司，再送其他；一般批次順序無所謂（api 檔只有一個）
    files = sorted(manifest["files"], key=lambda f: 0 if f["path"].startswith("api/companies") else 1)
    for f in files:
        rtype = f.get("record_type")
        data = src.bytes(f"batches/{name}/{f['path']}", f.get("sha256"))
        if rtype == "ir_document_file":
            target = volume_target(vol, f)
            write_bytes(target, data, dry)
            bronze.append(bronze_row(manifest, f, js(f), volume_path=target, now=now))   # payload = manifest 條目
            summary["landed"] += 1
            print(f"    落地 {f.get('category')}/{f.get('company_slug')}: {os.path.basename(f['path'])}")
            continue
        write_bytes(archive_target(vol, name, f["path"]), data, dry)
        summary["archived"] += 1
        text = data.decode("utf-8")
        bronze.append(bronze_row(manifest, f, text, volume_path=None, now=now))         # payload = JSON 全文
        if rtype == "api_payload":
            payloads.append((f, json.loads(text)))
    # manifest 本身也歸檔：Volume 那份給人看，bronze 那列給 SQL 查
    write_bytes(archive_target(vol, name, "manifest.json"),
                json.dumps(manifest, ensure_ascii=False, indent=1).encode("utf-8"), dry)

    # bronze → silver（只拿這一批的列；MERGE 冪等，就算後面 API 失敗、下次重跑也只是再算一次）
    summary["bronze_rows"] = write_bronze(settings, name, bronze)
    if summary["bronze_rows"]:
        df_batch = spark.table(table_name(settings, "bronze", "record")).filter(F.col("batch_id") == F.lit(name))
        summary["silver"] = write_silver(settings, df_batch)

    # 轉送系統 API；冪等鍵 = 批次 id + endpoint（主檔批次可能有多個 body）
    for f, payload in payloads:
        endpoint = endpoint_for(f)
        if not endpoint:
            raise ValueError(f"{name}/{f['path']}：不知道要送哪個 endpoint")
        idem_key = f"{name}/{endpoint}"
        status, body = api_post(settings, endpoint, payload, idem_key)
        if status is not None:
            summary["api"].append(api_result(endpoint=endpoint, idem_key=idem_key, rows=len(payload.get("rows", [])),
                                             http_status=status, body=body, now=datetime.now(UTC)))
    return summary


In [ ]:
# [c11] run_once
# 一次執行：健康檢查 → 找新批次 → 逐批處理（成功一批寫一列 batch_log、推一批游標）。


def run_once(settings: dict) -> dict:
    src = Source(settings["root_url"], settings["verify_ssl"])
    cursor = Cursor(settings["volume_root"])
    last_seq = int(cursor.load().get("last_seq") or 0)
    now = datetime.now(UTC)
    report = {"start_seq": last_seq, "processed": [], "alerts": [], "skipped_incomplete": []}
    print(f"root={settings['root_url']} volume={settings['volume_root']} 游標 seq={last_seq} "
          f"dry_run={settings['dry_run']} write_tables={settings['write_tables']}")

    # 1. 監控：health.json 太舊代表 cron 掛了。讀不到只記告警，不擋批次。
    try:
        health = src.json("health.json")
        if is_stale(health.get("generated_at"), now, settings["health_max_age_hours"]):
            report["alerts"].append(f"health.json 超過 {settings['health_max_age_hours']:.0f} 小時未更新"
                                    f"（generated_at={health.get('generated_at')}），爬蟲排程可能停了")
        if health.get("status") == "ALERT":
            report["alerts"].append(f"爬蟲端告警：{'; '.join(health.get('alerts') or [])}")
        vm_last = (health.get("batches") or {}).get("last_seq")
        print(f"health: {health.get('status')} generated_at={health.get('generated_at')} VM 最新批次 seq={vm_last}")
    except Exception as e:  # noqa: BLE001
        report["alerts"].append(f"讀不到 health.json：{e}")

    # 2. 找新批次：序號 > 游標，且有 _SUCCESS（沒有的代表還在搬，下次再看）
    names = src.listing("batches")
    todo = []
    for seq, name in pick_new_batches(names, last_seq):
        if src.exists(f"batches/{name}/_SUCCESS"):
            todo.append((seq, name))
        else:
            report["skipped_incomplete"].append(name)
    print(f"新批次 {len(todo)} 個" + (f"，另有 {len(report['skipped_incomplete'])} 個尚未搬完"
                                   if report["skipped_incomplete"] else ""))

    # 3. 逐批處理，成功一批推一批
    for seq, name in todo[: settings["max_batches"]]:
        try:
            summary = process_batch(src, settings, name, last_seq, datetime.now(UTC))
        except Exception as e:
            # 失敗也留一列 batch_log（讀得到 manifest 才寫得出來）；游標不動，重跑成功會覆蓋成 SUCCESS
            try:
                manifest = src.json(f"batches/{name}/manifest.json")
                write_batch_log(settings, batch_log_row(
                    manifest, status="FAILED", error=repr(e), job_run_id=settings["job_run_id"],
                    now=datetime.now(UTC)))
            except Exception as e2:  # noqa: BLE001
                print(f"    寫 batch_log FAILED 失敗：{e2}")
            raise
        write_batch_log(settings, batch_log_row(
            summary["manifest"], status="SUCCESS", landed=summary["landed"], archived=summary["archived"],
            bronze_rows=summary["bronze_rows"], api_results=summary["api"],
            job_run_id=settings["job_run_id"], now=datetime.now(UTC)))
        if not settings["dry_run"]:
            cursor.save(seq, name)
        last_seq = seq
        report["processed"].append({k: v for k, v in summary.items() if k != "manifest"})

    report["end_seq"] = last_seq
    print(f"完成：處理 {len(report['processed'])} 批，游標 {report['start_seq']} → {last_seq}")
    if report["alerts"]:
        print("告警：\n  - " + "\n  - ".join(report["alerts"]))
    return report


In [ ]:
# [c12] rebuild_silver
# 從整張 bronze 重算 silver：silver 改欄位 / 改 transform / 懷疑資料壞掉時用（rebuild_silver = true）。
# 不動 bronze、不動游標、不打 API。同鍵取最新 seq（latest_per_key），所以重算結果 = 逐批增量的結果。


def rebuild_silver(settings: dict) -> dict[str, int]:
    df_all = spark.table(table_name(settings, "bronze", "record"))
    print(f"rebuild silver from {table_name(settings, 'bronze', 'record')}")
    return write_silver(settings, df_all)


In [ ]:
# [c20] main
# job 進入點。rebuild_silver = true 只重算 silver；否則消費新批次。有告警就 raise，讓 Databricks job 顯示失敗、通知才會發。
if settings["rebuild_silver"]:
    print("silver 重算結果：", rebuild_silver(settings))
else:
    report = run_once(settings)
    if report["alerts"]:
        raise RuntimeError("; ".join(report["alerts"]))


In [ ]:
# [c30] check_tables
# 收尾：印最近 5 批的處理紀錄與各 silver 表列數（都是小表，limit / count 可接受）。dry_run 或 write_tables=false 時略過。
if settings["write_tables"] and not settings["dry_run"]:
    t = table_name(settings, "bronze", "batch_log")
    for r in spark.table(t).orderBy("seq", ascending=False).limit(5).collect():
        print(f"{r.batch_id}  {r.status:<8} kind={r.kind} landed={r.landed_files} archived={r.archived_files} "
              f"bronze={r.bronze_rows} api={len(r.api_results or [])} processed_at={r.processed_at}")
    for short in SILVER_TRANSFORMS:
        t = table_name(settings, "silver", short)
        print(f"{t}: rows={spark.table(t).count()}")
